In [ ]:
#!pip install openai

In [63]:
import json
import pandas as pd
import os
from dotenv import load_dotenv
from openai import OpenAI

In [64]:
OPENAI_MODEL = "gpt-4o-mini" #gpt-5-mini

In [ ]:
SYSTEM_PROMPT = (
    "Sei un estrattore di informazioni. "
    "Devi rispondere SOLO in formato json valido (oggetto JSON), senza testo extra, "
    "senza markdown e senza code fences. "
    "La risposta deve essere un unico oggetto json."    
)

In [ ]:
USER_INSTRUCTIONS = """Estrai l'elenco strutturato dei centri/sportelli/case citati nel testo qui sotto.
Regole:
- 'tipo' ∈ {Centro Antiviolenza, Sportello collegato, Casa Rifugio, Altro}
- Compila comuni/indirizzi solo se esplicitamente presenti.
- Indica in 'ente_capofila' se dal testo emerge (es. “Comune di Bra”).
- In 'note' aggiungi contesto utile (es. “sportelli decentrati del CAV n.10/A del Cuneese”, “collegamento al 1522”, “nuovo centro”).
Testo:
"""

In [ ]:
USER_INSTRUCTIONS = """Estrai dal testo un elenco strutturato di tutte le entità citate (centri, sportelli, case rifugio, associazioni, enti gestori, consorzi, ASL, comuni, province).
Regole:
- Ogni entità deve avere:
  - 'nome': nome completo come appare nel testo
  - 'tipo': uno tra {Centro Antiviolenza, Sportello collegato, Casa Rifugio, Associazione di volontariato, Associazione di promozione sociale, Ente territoriale, Consorzio socio-assistenziale, ASL, Altro}
  - 'comune': compila se esplicitamente presente
  - 'indirizzo': compila se esplicitamente presente
  - 'ente_capofila': se dal testo emerge chiaramente
  - 'note': aggiungi contesto utile (es. “sportelli decentrati del CAV n.10/A del Cuneese”, “collegamento al 1522”, “in coordinamento con altre associazioni”)
- Non perdere associazioni o enti partner: vanno inclusi anche se non sono centri/sportelli.
- Mantieni sempre gli indirizzi e i comuni se presenti nel testo.
- Rispondi SOLO in un unico oggetto JSON valido con una lista sotto la chiave 'entities'.
Testo:
"""


In [ ]:
USER_INSTRUCTIONS = """Analizza il testo ed estrai tutte le entità rilevanti (centri, sportelli, case rifugio, associazioni, consorzi, ASL, enti territoriali).
Regole di estrazione e formato:
- Restituisci un UNICO oggetto JSON con una lista sotto la chiave 'entities'.
- Ogni elemento di 'entities' deve contenere i campi:
  - 'nome' (stringa): nome completo come appare nel testo.
  - 'tipo' (stringa): uno tra {
      Centro Antiviolenza, Sportello collegato, Casa Rifugio,
      Associazione di volontariato, Associazione di promozione sociale,
      Consorzio socio-assistenziale, ASL, Ente territoriale, Altro
    }.
  - 'comune' (stringa): valorizza SOLO se esplicitamente presente o ricavabile da normalizzazione geografica.
  - 'indirizzo' (stringa): valorizza SOLO se esplicitamente presente (via/piazza/numero civico).
  - 'ente_capofila' (stringa): valorizza se dal testo emerge esplicitamente chi è l’ente capofila/gestore.
  - 'note' (stringa): dettagli aggiuntivi utili (es. “sportelli decentrati del CAV n.10/A del Cuneese”, “collegamento al 1522”, “nuovo centro”, “in coordinamento con …”).

Regole di copertura (evita perdite di informazione):
- NON limitarti a centri/sportelli/case: includi anche associazioni, consorzi, ASL ed enti territoriali citati come partner o gestori.
- Mantieni SEMPRE 'comune' e 'indirizzo' quando sono presenti nel testo (anche se collegati a un ente partner).
- Se un soggetto è indicato come “ente capofila” o “gestore”, compila 'ente_capofila' con il SUO nome (oltre a riportarlo in 'nome'); usa 'note' per i dettagli extra.
- Se il testo indica “sportelli decentrati” o collegamenti a un CAV esistente, riportalo nelle 'note' dell’ente/centro pertinente.

Normalizzazioni e disambiguazioni:
- Normalizza riferimenti geografici generici in toponimi espliciti quando chiaramente deducibili:
  - “del Cuneese” → comune “Cuneo”.
  - Se è indicata una provincia/area ma non il comune, lascia 'comune' vuoto.
- Conserva e normalizza sigle e abbreviazioni note: ASL (azienda sanitaria), Ass., Ass.ne, Univ., Pref.
- Non inventare dati: se un campo non è presente, lascialo come stringa vuota "".

Deduplicazione e coalescenza:
- Unisci duplicati evidenti dello stesso ente/centro (stesso nome o varianti minime); preferisci la forma più completa.
- Se lo stesso ente ha più ruoli (es. partner e gestore), usa un UNICO record e descrivi i ruoli in 'note'.

Output:
- Rispondi SOLO con un oggetto JSON valido (nessun testo extra, nessun markdown, nessun code fence).
Testo:
"""


In [ ]:
USER_INSTRUCTIONS = """
Analizza il testo fornito in formato json ed estrai per ogni entità le chiavi:
- nome, tipo, comune, indirizzo, ente_capofila, note


Regole:

Poi applica rigorosamente:

1) Se "indirizzo" è vuoto:
   a) Se "nome" è valorizzato → cerca un indirizzo valido per quel nome.
      - Se trovato: valorizza "indirizzo" e:
        • Se ricavi un comune dall’indirizzo:
            - Se "comune" è vuoto → impostalo.
            - Se "comune" esiste e differisce → registra conflitto in conflitti.comune.
        • Se disponibili, valorizza "provincia" e "regione"; se non presenti nell’indirizzo, deducile dal comune.
   b) Altrimenti, se "comune" è valorizzato → cerca un indirizzo coerente in quel comune per l’ente.
      - Se trovato: valorizza "indirizzo" e crea/valorizza "provincia" e "regione" (dedotte se necessario).

2) Se "indirizzo" è valorizzato:
   - Estrai/deduci "comune" dall’indirizzo:
     • Se "comune" è vuoto → impostalo.
     • Se "comune" esiste e differisce → registra conflitto in conflitti.comune.
   - Crea/valorizza "provincia" e "regione" se presenti; altrimenti deducile dal comune.

3) Normalizza:
   - Pulisci spazi/virgolette; sigle in MAIUSCOLO (ASL, ODV, APS, ONLUS).
   - Uniforma "tipo" alle categorie: Ente territoriale, ASL, Consorzio socio-assistenziale,
     Associazione di volontariato (OdV), Associazione di promozione sociale (APS), Altro.

4) De-duplicazione:
   - Duplicato = stesso "nome" e stesso "comune" (case-insensitive, ignora punteggiatura).
   - Mantieni l’indirizzo più specifico; alternative in "note".

5) Provenienza e qualità:
   - Se un valore è dedotto → imposta fonte_* = "dedotto"; dal testo → "testo"; da conoscenza → "conoscenza".
   - Se c’è conflitto sul comune → compila "conflitti.comune" con "dato" e "trovato".
   - Aggiungi "confidence" ∈ [0.0, 1.0] seguendo le linee guida del SYSTEM_PROMPT.

6) Output:
   - Restituisci SOLO JSON valido come da schema indicato solo JSON valido (UTF-8) con la seguente struttura:

{
  "file": "<nome_file_input_o_placeholder>",
  "entities": [
    {
      "nome": "",
      "tipo": "",
      "comune": "",
      "indirizzo": "",
      "ente_capofila": "",
      "note": "",
      "provincia": "",
      "regione": "",
      "fonte_indirizzo": "testo|dedotto|conoscenza",
      "fonte_provincia": "testo|dedotto|conoscenza",
      "fonte_regione": "testo|dedotto|conoscenza",
      "conflitti": {
        "comune": { "dato": "", "trovato": "" }
      },
      "confidence": 0.0
    }
  ]
}

Includi provincia, regione, fonte_*, conflitti, confidence solo se pertinenti.
Niente testo fuori dal JSON.
"""


In [69]:
# ner_prompt_config.py
# -*- coding: utf-8 -*-
"""
Configurazione prompt per LLM/NER:
- SYSTEM_PROMPT
- USER_INSTRUCTIONS
- JSON_SCHEMA_STR
- Helper build_user_message()
- EXAMPLE_MESSAGES
"""

# =========================
# 1) SYSTEM PROMPT
# =========================
SYSTEM_PROMPT = r"""
Sei un agente specializzato in estrazione e arricchimento di entità da testi istituzionali italiani (protocolli, convenzioni, atti, delibere).
Produci esclusivamente JSON valido secondo lo schema indicato, senza testo extra.

Obiettivo
---------
Per ogni entità individuata nel testo, estrai le chiavi:
- nome, tipo, comune, indirizzo, ente_capofila, note

e arricchisci con:
- provincia, regione (se ricavabili)

Aggiungi metadati facoltativi:
- fonte_indirizzo, fonte_provincia, fonte_regione ∈ {testo, dedotto, conoscenza}
- conflitti (solo se presenti)
- confidence ∈ [0.0, 1.0]

Ontologia e normalizzazione
---------------------------
1) tipo ∈ {
   "Ente territoriale",
   "ASL",
   "Consorzio socio-assistenziale",
   "Associazione di volontariato (OdV)",
   "Associazione di promozione sociale (APS)",
   "Altro"
}
2) nome: mantieni grafia ufficiale; sigle in MAIUSCOLO (ASL, ODV, APS, ONLUS). Pulisci spazi doppi/virgolette ornamentali.
3) indirizzo (valido se contiene toponimo + nome via/piazza e preferibilmente civico).
   Toponimi ammessi (case-insensitive): Via|Viale|V\.le|Vicolo|Corso|C\.so|Piazza|P\.zza|Largo|Piazzale|P\.le|Strada|Borgo|Traversa|Località|Frazione|Regione.
   Accetta CAP (\b\d{5}\b) e sigla provincia tra parentesi (es. (BI)).
4) Comune/Provincia/Regione:
   - Se presenti nell’indirizzo → estrai.
   - Se assenti ma il comune è noto → deduci provincia e regione dal comune.
   - Se trovi solo sigla provincia → valorizza provincia se noto altrimenti conserva sigla e riduci confidence.

Pipeline (ordine obbligatorio)
------------------------------
1) Preprocessa: pulisci whitespace e apostrofi; unisci righe spezzate di un indirizzo.
2) Estrai entità: crea un record per ogni organizzazione (Comune di…, Provincia di…, Ambito…, ASL…, Consorzio…, Associazione…, Procura…, Università…, Dipartimento…, Cooperativa…).
3) Gestione indirizzo:
   Caso 1 — indirizzo vuoto:
     1a) Se nome è valorizzato → cerca indirizzo valido per quel nome nel contesto.
     1b) Altrimenti, se comune è valorizzato → cerca indirizzo coerente in quel comune.
   Caso 2 — indirizzo presente:
     - Estrai/deduci comune dall’indirizzo; se diverso dal campo comune, registra conflitto.
     - Estrai/deduci provincia e regione.
4) Fonti:
   - Dal testo → "testo"
   - Deduci da altro campo → "dedotto"
   - Da conoscenza esterna → "conoscenza"
5) Conflitti:
   - Se comune (record) ≠ comune (da indirizzo) → conflitti.comune = { "dato": "...", "trovato": "..." }
6) De-duplicazione:
   - Duplicato = stesso nome + stesso comune
   - Mantieni l’indirizzo più specifico, sposta alternative in note.
7) Confidence:
   - Base 0.50 se nome+tipo trovati
   - +0.15 indirizzo con civico; +0.10 senza civico ma univoco
   - +0.10 coerenza comune–provincia–regione
   - −0.20 conflitto comune; −0.10 provincia/regione parziali
   - Clampa a [0.0, 1.0]

Output (solo JSON)
------------------
{
  "file": "<nome_file_input_o_placeholder>",
  "entities": [
    {
      "nome": "",
      "tipo": "",
      "comune": "",
      "indirizzo": "",
      "ente_capofila": "",
      "note": "",
      "provincia": "",
      "regione": "",
      "fonte_indirizzo": "testo|dedotto|conoscenza",
      "fonte_provincia": "testo|dedotto|conoscenza",
      "fonte_regione": "testo|dedotto|conoscenza",
      "conflitti": {
        "comune": { "dato": "", "trovato": "" }
      },
      "confidence": 0.0
    }
  ]
}
""".strip()

# =============================
# 2) USER INSTRUCTIONS
# =============================
USER_INSTRUCTIONS = r"""
Analizza il testo fornito ed estrai per ogni entità le chiavi:
- nome, tipo, comune, indirizzo, ente_capofila, note

Poi applica rigorosamente:

1) Se "indirizzo" è vuoto:
   a) Se "nome" è valorizzato → cerca un indirizzo valido per quel nome.
   b) Altrimenti, se "comune" è valorizzato → cerca un indirizzo coerente in quel comune.
   In entrambi i casi → valorizza anche provincia e regione (dedotte se necessario).

2) Se "indirizzo" è valorizzato:
   - Estrai/deduci "comune"; se differisce → registra conflitto.
   - Valorizza "provincia" e "regione" (o deducile dal comune).

3) Normalizza:
   - Pulisci spazi/virgolette; sigle in MAIUSCOLO.
   - Uniforma "tipo" alle categorie predefinite. 
   - Normalizza sempre provincia al nome esteso (es. Cuneo) e, se disponibile, aggiungi anche provincia_sigla (es. CN). Non usare solo la sigla in provincia.
  -  Non impostare confidence a 1.0 se manca il CAP o se provincia/regione sono dedotte; in tali casi, max 0.95.
  -  file in output deve essere identico al nome file dichiarato nel messaggio utente.

4) De-duplicazione:
   - Stesso nome+comune → unisci; tieni indirizzo più specifico; altri in note.

5) Provenienza e qualità:
   - Fonte dal testo → "testo"
   - Fonte dedotta → "dedotto"
   - Fonte da conoscenza → "conoscenza"
   - Conflitti nel campo "conflitti"
   - Aggiungi "confidence" ∈ [0.0, 1.0]

6) Output:
   - Solo JSON valido come nello schema del SYSTEM_PROMPT.
""".strip()

# =========================
# 3) JSON Schema (opzionale)
# =========================
JSON_SCHEMA_STR = r"""
{
  "type": "object",
  "required": ["file", "entities"],
  "properties": {
    "file": { "type": "string" },
    "entities": {
      "type": "array",
      "items": {
        "type": "object",
        "required": ["nome", "tipo", "comune", "indirizzo", "ente_capofila", "note"],
        "properties": {
          "nome": { "type": "string" },
          "tipo": { "type": "string" },
          "comune": { "type": "string" },
          "indirizzo": { "type": "string" },
          "ente_capofila": { "type": "string" },
          "note": { "type": "string" },
          "provincia": { "type": "string" },
          "regione": { "type": "string" },
          "fonte_indirizzo": { "type": "string" },
          "fonte_provincia": { "type": "string" },
          "fonte_regione": { "type": "string" },
          "conflitti": { "type": "object" },
          "confidence": { "type": "number" }
        }
      }
    }
  }
}
""".strip()

# =====================================
# 4) Helper per creare il messaggio utente
# =====================================
def build_user_message(raw_text: str, file_name: str = "input.txt") -> str:
    """
    Prepara le istruzioni utente concatenando il testo e il nome file.
    """
    return (
        f"FILE: {file_name}\n\n"
        f"ISTRUZIONI:\n{USER_INSTRUCTIONS}\n\n"
        f"TESTO DA ANALIZZARE:\n---\n{raw_text}\n---\n"
    )

# =====================================
# 5) Esempio messages per API chat
# =====================================
EXAMPLE_MESSAGES = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": build_user_message("<<testo da analizzare>>",
                                                   file_name="documento.txt")}
]


In [ ]:
import os
import json

cartella_txt = "script/txt/01"

# 1️⃣ Carica tutti i file .txt in un array
testi = []
file_txt = [f for f in os.listdir(cartella_txt) if f.endswith(".txt")]

for nome_file in file_txt:
    percorso = os.path.join(cartella_txt, nome_file)
    with open(percorso, "r", encoding="utf-8") as f:
        testo = f.read()
        testi.append({"nome_file": nome_file, "contenuto": testo})

print(f"📂 Caricati {len(testi)} file di testo.")

In [67]:
# Chiave API
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

In [ ]:
risultati = []

for item in testi:
    try:
        resp = client.responses.create(
            model=OPENAI_MODEL,
            instructions=SYSTEM_PROMPT,
            input=USER_INSTRUCTIONS + item["contenuto"] + "\n---"
        )
        data = json.loads(resp.output_text)
        risultati.append({
            "file": item["nome_file"],
            "risultato": data
        })
    except Exception as e:
        print(f"❌ Errore con {item['nome_file']}: {e}")
        risultati.append({
            "file": item["nome_file"],
            "risultato": None
        })

# 3️⃣ Stampa i risultati (primi 2 come esempio)
print(json.dumps(risultati[:2], indent=4, ensure_ascii=False))

In [70]:
import json
import re

# === CONFIG ===
PERCORSO_JSON = "script/output/json/01_risultati.json"
OPENAI_MODEL = "gpt-4o-mini"  # o quello che stai usando



# Helper: costruisce il messaggio utente includendo il nome file .txt
def build_user_message(entity: dict, file_txt: str) -> str:
    """
    Prepara il contenuto utente con:
    - FILE: <nomefile.txt>
    - ISTRUZIONI: <USER_INSTRUCTIONS>
    - ENTITY: <json dell'entità di partenza>
    """
    ent_str = json.dumps(entity, ensure_ascii=False)
    return (
        f"FILE: {file_txt}\n\n"
        f"ISTRUZIONI:\n{USER_INSTRUCTIONS}\n\n"
        f"ENTITY DI PARTENZA:\n{ent_str}\n---"
    )

def estrai_json_da_testo(testo: str):
    """
    Prova a estrarre un oggetto JSON dal testo:
    1) blocchi in ```json ... ```
    2) blocchi in ``` ... ```
    3) prima porzione che sembra un oggetto { ... }
    Ritorna: (dict | None, errore | None)
    """
    if testo is None:
        return None, "output_text è None"

    # 1) ```json ... ```
    m = re.search(r"```json\s*(\{.*?\})\s*```", testo, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1)), None
        except Exception as e:
            return None, f"JSONDecodeError in blocco ```json```: {e}"

    # 2) ``` ... ```
    m = re.search(r"```\s*(\{.*?\})\s*```", testo, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1)), None
        except Exception:
            pass  # continua con fallback

    # 3) Fallback: prima { ... } “grande”
    start = testo.find('{')
    end = testo.rfind('}')
    if start != -1 and end != -1 and end > start:
        candidato = testo[start:end+1]
        try:
            return json.loads(candidato), None
        except Exception as e:
            return None, f"JSONDecodeError nel fallback: {e}"

    return None, "Nessun oggetto JSON trovato nel testo"


# --- Carica input ---
with open(PERCORSO_JSON, "r", encoding="utf-8") as f:
    dati = json.load(f)

risultati = []

# ⚠️ Assumo che tu abbia un client già istanziato (es. client = OpenAI()) e SYSTEM_PROMPT/USER_INSTRUCTIONS reali
for item in dati:
    file_txt = item.get("file", "")  # <-- QUI c'è il nome del file .txt da propagare
    entities = item.get("risultato", {}).get("entities", [])

    for ent in entities:
        try:
            # Log di debug: l'ENTITY iniziale
            print(">> ENTITY:", json.dumps(ent, ensure_ascii=False))

            # ===== CHIAMATA MODELLO =====
            # Se usi responses API tipo "client.responses.create"
            resp = client.responses.create(
                model=OPENAI_MODEL,
                instructions=SYSTEM_PROMPT,
                input=build_user_message(ent, file_txt)
            )

            # Estrai testo (adatta al tuo SDK)
            output_text = getattr(resp, "output_text", None)
            if output_text is None:
                try:
                    output_text = str(resp)
                except Exception:
                    output_text = ""

            data, parse_err = estrai_json_da_testo(output_text)

            if parse_err or not isinstance(data, dict):
                print(f"!! Parsing fallito per {file_txt} - entity {ent.get('nome','')}: {parse_err}")
                print(">> OUTPUT GREZZO:", (output_text[:500] + "…") if len(output_text) > 500 else output_text)
                risultati.append({
                    "file-json": file_txt,
                    "file-txt": file_txt,  # mostriamo comunque il txt corrente
                    "nome": ent.get("nome", ""),
                    "risultato": None,
                    "errore": f"parse_error: {parse_err or 'json non dict'}"
                })
                continue

            # ===== FORZA/PROPAGA NOME FILE NELL'OUTPUT =====
            # Se il modello ha messo "placeholder" o ha omesso "file", sostituisci con file_txt
            model_file = data.get("file")
            if (not model_file) or (isinstance(model_file, str) and model_file.strip().lower() in {"placeholder", "input_placeholder"}):
                data["file"] = file_txt

            # (Facoltativo) Se l'output non contiene affatto la chiave "file", la creiamo:
            if "file" not in data:
                data["file"] = file_txt

            print("outjson", data)

            risultati.append({
                "file-json": file_txt,    # file sorgente del JSON aggregato
                "file-txt": file_txt,     # nome del txt visualizzato
                "nome": ent.get("nome", ""),
                "risultato": data
            })

        except Exception as e:
            print(f"❌ Errore con {file_txt} - entity {ent.get('nome', '')}: {e}")
            risultati.append({
                "file-json": file_txt,
                "file-txt": file_txt,
                "nome": ent.get("nome", ""),
                "risultato": None,
                "errore": str(e)
            })

# Stampa anteprima
print(json.dumps(risultati[:5], indent=4, ensure_ascii=False))




>> ENTITY: {"nome": "Comune di Bra", "tipo": "Ente territoriale", "comune": "Bra", "indirizzo": "Piazza Caduti per la Libertà n. 14", "ente_capofila": "Comune di Bra", "note": "Ente capofila e gestore dei servizi socio-assistenziali."}
outjson {'file': '01_2406_rta_01_240904110730_4166---protocollocontuttelefirme(1).txt', 'entities': [{'nome': 'Comune di Bra', 'tipo': 'Ente territoriale', 'comune': 'Bra', 'indirizzo': 'Piazza Caduti per la Libertà n. 14', 'ente_capofila': 'Comune di Bra', 'note': 'Ente capofila e gestore dei servizi socio-assistenziali.', 'provincia': 'Cuneo', 'regione': 'Piemonte', 'fonte_indirizzo': 'testo', 'fonte_provincia': 'dedotto', 'fonte_regione': 'dedotto', 'conflitti': {}, 'confidence': 0.95}]}
>> ENTITY: {"nome": "Consorzio Socio Assistenziale Alba Langhe Roero", "tipo": "Consorzio socio-assistenziale", "comune": "Alba", "indirizzo": "Via A. Diaz n. 8", "ente_capofila": "", "note": "Partner nel protocollo."}
outjson {'file': '01_2406_rta_01_240904110730_416

In [71]:
import json
import os

# 📂 Cartella di output
output_dir = "script/output/json"
os.makedirs(output_dir, exist_ok=True)

# 📄 Salva tutti i risultati in un unico file
output_file = os.path.join(output_dir, "01.1_risultati.json")

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(risultati, f, indent=4, ensure_ascii=False)

print(f"✅ Risultati salvati in: {output_file}")


✅ Risultati salvati in: script/output/json\01.1_risultati.json
